In [3]:
%load_ext autoreload
%autoreload 2

# autoreload re-imports src.* modules from disk before every cell run, so
# edits to wall_extraction.py/door_extraction.py/window_extraction.py/
# run_mask_to_vector.py etc. take effect without restarting the kernel.
# Exception: the run3 model/checkpoint itself is still cached in
# _MODEL_CACHE (see vectorize_image below) - that's intentional, only the
# vectorization code should re-import on every run, not reload the model.

# Run3 single-image vectorization

Set `INPUT_IMAGE_PATH` in the last cell to any floor plan image, then **Run All Cells**.

This runs: clean input -> segformer_b0_run3 CNN segmentation -> v008 point-graph vectorization, and prints the path to the generated `..._vector.svg` (saved next to the input image, along with the cleaned input, the CNN prediction preview, a debug overlay, and metrics.json).

In [4]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import torch
from PIL import Image, ImageOps
from torchvision import transforms


def _find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "vectorization").exists() and (candidate / "configs").exists():
            return candidate
    notebook_parent = Path(__file__).resolve().parents[1] if "__file__" in globals() else here
    return notebook_parent


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.checkpointing import load_checkpoint
from src.models import FloorplanSegModel, build_backbone, build_decoder
from src.train_segmentation import _mask_to_rgb
from src.vectorization.run_mask_to_vector import (
    _scale_info_from_config,
    load_config,
    process_single,
)


IMAGE_SIZE = 512
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / "segformer_b0_run3" / "best.pt"
VECTORIZATION_CONFIG_PATH = PROJECT_ROOT / "configs" / "vectorization_v008.yaml"

_MODEL_CACHE = None


def _clean_image_for_model(image_path: Path, cleaned_path: Path) -> Image.Image:
    """Load and normalize the input raster into a clean RGB image for inference."""
    with Image.open(image_path) as img:
        img = ImageOps.exif_transpose(img)
        if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
            rgba = img.convert("RGBA")
            bg = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
            img = Image.alpha_composite(bg, rgba).convert("RGB")
        else:
            img = img.convert("RGB")
    cleaned_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(cleaned_path)
    return img


def _load_run3_model():
    global _MODEL_CACHE
    if _MODEL_CACHE is not None:
        return _MODEL_CACHE
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"Run3 checkpoint not found: {CHECKPOINT_PATH}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = build_backbone(variant="segformer_b0", pretrained=True)
    decoder = build_decoder(variant="segformer_b0", num_classes=7, output_size=IMAGE_SIZE)
    payload = load_checkpoint(CHECKPOINT_PATH, decoder, device=device)
    model = FloorplanSegModel(backbone=backbone, decoder=decoder).eval().to(device)
    _MODEL_CACHE = (model, device, payload)
    return _MODEL_CACHE


def vectorize_image(image_path):
    """Run cleaning -> run3 CNN segmentation -> v008 vectorization for one image.

    Args:
        image_path: Path to the input raster image.

    Saves beside the input image:
        <stem>_cleaned.png
        <stem>_prediction.png
        <stem>_vector.svg
        debug_overlay.png
        metrics.json

    Returns:
        Path to the generated SVG.
    """
    image_path = Path(image_path).expanduser().resolve()
    if not image_path.exists():
        raise FileNotFoundError(f"Input image not found: {image_path}")

    out_dir = image_path.parent
    stem = image_path.stem
    cleaned_path = out_dir / f"{stem}_cleaned.png"
    prediction_path = out_dir / f"{stem}_prediction.png"
    svg_name = f"{stem}_vector.svg"
    svg_path = out_dir / svg_name

    cleaned_img = _clean_image_for_model(image_path, cleaned_path)

    model, device, payload = _load_run3_model()
    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    x = transform(cleaned_img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x).argmax(dim=1)[0].cpu().numpy()
    Image.fromarray(_mask_to_rgb(pred)).save(prediction_path)

    vcfg = load_config(VECTORIZATION_CONFIG_PATH)
    scale_info = _scale_info_from_config(vcfg)
    process_single(prediction_path, vcfg, scale_info, out_dir, output_filename=svg_name)

    # process_single writes fixed debug names. Keep them and also create stem-scoped copies.
    fixed_debug = out_dir / "debug_overlay.png"
    fixed_metrics = out_dir / "metrics.json"
    if fixed_debug.exists():
        shutil.copy2(fixed_debug, out_dir / f"{stem}_debug_overlay.png")
    if fixed_metrics.exists():
        shutil.copy2(fixed_metrics, out_dir / f"{stem}_metrics.json")

    summary = {
        "input": str(image_path),
        "cleaned_input": str(cleaned_path),
        "prediction": str(prediction_path),
        "svg": str(svg_path),
        "debug_overlay": str(out_dir / f"{stem}_debug_overlay.png"),
        "metrics": str(out_dir / f"{stem}_metrics.json"),
        "device": str(device),
        "checkpoint_epoch": payload.get("epoch"),
        "checkpoint_arch_version": payload.get("arch_version"),
    }
    print(json.dumps(summary, indent=2))
    return svg_path

In [5]:
# <<< EDIT THIS, then Run All Cells >>>
INPUT_IMAGE_PATH = r"C:\Users\kdgki\Desktop\MSCDP\Projects\neural_floorplan\outputs\vectorization\v008\iteration5_run3\sample_005\image.png"

vectorize_image(INPUT_IMAGE_PATH)

Loading weights: 100%|██████████| 192/192 [00:00<00:00, 20172.50it/s]
[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Processing: image_prediction.png
    walls=67, windows=0, doors=1, rejected=107, scale=estimated
    -> C:\Users\kdgki\Desktop\MSCDP\Projects\neural_floorplan\outputs\vectorization\v008\iteration5_run3\sample_005\image_vector.svg
{
  "input": "C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\v008\\iteration5_run3\\sample_005\\image.png",
  "cleaned_input": "C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\v008\\iteration5_run3\\sample_005\\image_cleaned.png",
  "prediction": "C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\v008\\iteration5_run3\\sample_005\\image_prediction.png",
  "svg": "C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\v008\\iteration5_run3\\sample_005\\image_vector.svg",
  "debug_overlay": "C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\v008\\iteration5_run3\\sample_005\\image_de

WindowsPath('C:/Users/kdgki/Desktop/MSCDP/Projects/neural_floorplan/outputs/vectorization/v008/iteration5_run3/sample_005/image_vector.svg')